# 04. CLRKDNet ONNX Runtime Latency 검증 Runner

이 노트북의 질문은 다음과 같다.

> ONNX로 변환한 CLRKDNet이 Raspberry Pi 5 CPU에서 lane following에 쓸 만큼 빠르게 도는가?

ONNX export가 성공했다는 것은 "모델 파일을 만들 수 있다"는 뜻이고, Pi FPS 검증은 "그 모델을 실제 자동차에서 쓸 수 있다"는 뜻에 가깝다. 둘은 다르다.

In [2]:
from pathlib import Path
import sys
import subprocess
import textwrap
import re
from IPython.display import Markdown, Image, display

ROOT = Path.cwd()
# Support both launch locations: architecture_validation/ and architecture_validation/notebooks/.
if ROOT.name == 'notebooks':
    ROOT = ROOT.parent
elif not (ROOT / 'scripts').exists() and (ROOT.parent / 'scripts').exists():
    ROOT = ROOT.parent
SCRIPTS = ROOT / 'scripts'
MODELS = ROOT / 'models'
OUTPUTS = ROOT / 'outputs'
OUTPUTS.mkdir(exist_ok=True)
print(f"validation root: {ROOT}")
print(f"scripts dir: {SCRIPTS}")
print(f"models dir: {MODELS}")
print(f"outputs dir: {OUTPUTS}")
print(f"python: {sys.executable}")

def show_lines(path, start=None, end=None):
    path = Path(path)
    lines = path.read_text(encoding="utf-8").splitlines()
    start = 1 if start is None else start
    end = len(lines) if end is None else end
    for lineno in range(start, min(end, len(lines)) + 1):
        print(f"{lineno:04d}: {lines[lineno - 1]}")

def run_cmd(args, check=False):
    print("$ " + " ".join(str(a) for a in args))
    result = subprocess.run(args, text=True, capture_output=True, cwd=str(ROOT))
    if result.stdout:
        print(result.stdout)
    if result.stderr:
        print("[stderr]")
        print(result.stderr)
    if check and result.returncode != 0:
        raise RuntimeError(f"command failed with code {result.returncode}")
    print(f"returncode={result.returncode}")
    return result

SCRIPT = SCRIPTS / '03_lane_onnx_bench.py'
MODEL = MODELS / 'clrkdnet_resnet18_culane.onnx'
assert SCRIPT.exists(), SCRIPT
assert MODEL.exists(), MODEL
print(SCRIPT)
print(MODEL)

validation root: /home/pi/AI_CAR/architecture_validation/lane_model
scripts dir: /home/pi/AI_CAR/architecture_validation/lane_model/scripts
models dir: /home/pi/AI_CAR/architecture_validation/lane_model/models
outputs dir: /home/pi/AI_CAR/architecture_validation/lane_model/outputs
python: /home/pi/AI_CAR/env/bin/python3
/home/pi/AI_CAR/architecture_validation/lane_model/scripts/03_lane_onnx_bench.py
/home/pi/AI_CAR/architecture_validation/lane_model/models/clrkdnet_resnet18_culane.onnx


## 코드 읽기 1 — ONNX Runtime thread 설정

여기서 `threads`는 프로그램의 lane/sign thread가 아니다.

모델을 한 번 추론할 때 ONNX Runtime이 CPU 내부 thread를 몇 개 쓸지 정하는 값이다. Pi에서는 thread를 무조건 많이 쓴다고 빨라지지 않으므로 직접 sweep해야 한다.

In [3]:
show_lines(SCRIPT, 61, 66)

0061: def build_session(model_path: Path, threads: int) -> ort.InferenceSession:
0062:     options = ort.SessionOptions()
0063:     if threads > 0:
0064:         options.intra_op_num_threads = threads
0065:         options.inter_op_num_threads = 1
0066:     return ort.InferenceSession(str(model_path), sess_options=options, providers=["CPUExecutionProvider"])


## 코드 읽기 2 — dummy / image / camera 모드

- `--dummy`: 순수 모델 추론 속도만 봄.
- `--image`: 파일 이미지 전처리 + 추론.
- `--camera`: 실제 카메라 frame + 전처리 + 추론.

Log 03에서는 dummy와 camera 결과를 반드시 구분해서 적어야 한다.

In [4]:
show_lines(SCRIPT, 72, 91)

0072:     parser.add_argument("--image", type=Path, default=None)
0073:     parser.add_argument("--camera", action="store_true")
0074:     parser.add_argument("--dummy", action="store_true")
0075:     parser.add_argument("--width", type=int, default=1296)
0076:     parser.add_argument("--height", type=int, default=972)
0077:     parser.add_argument("--input-width", type=int, default=800)
0078:     parser.add_argument("--input-height", type=int, default=320)
0079:     parser.add_argument("--frames", type=int, default=200)
0080:     parser.add_argument("--warmup", type=int, default=20)
0081:     parser.add_argument("--crop-top", type=int, default=None)
0082:     parser.add_argument("--color-order", choices=["rgb", "bgr"], default="rgb")
0083:     parser.add_argument("--no-flip-180", action="store_true", help="Disable default 180-degree camera flip")
0084:     parser.add_argument("--threads", type=int, default=0)
0085:     args = parser.parse_args()
0086: 
0087:     if not args.model.exis

## 1차 실행 — Dummy thread sweep

먼저 카메라 없이 모델 자체의 속도를 본다. 이전 러프 측정에서는 4 threads가 가장 빨랐다.

In [5]:
THREADS = [1, 2, 4, 6, 8]
FRAMES = "100"
WARMUP = "10"

dummy_results = {}
for th in THREADS:
    print("\n" + "=" * 80)
    print(f"threads={th}")
    cmd = [
        sys.executable, str(SCRIPT),
        "--model", str(MODEL),
        "--dummy",
        "--frames", FRAMES,
        "--warmup", WARMUP,
        "--threads", str(th),
    ]
    result = run_cmd(cmd)
    dummy_results[th] = result.stdout


threads=1
$ /home/pi/AI_CAR/env/bin/python3 /home/pi/AI_CAR/architecture_validation/lane_model/scripts/03_lane_onnx_bench.py --model /home/pi/AI_CAR/architecture_validation/lane_model/models/clrkdnet_resnet18_culane.onnx --dummy --frames 100 --warmup 10 --threads 1
model: /home/pi/AI_CAR/architecture_validation/lane_model/models/clrkdnet_resnet18_culane.onnx
input: image ['batch', 3, 320, 800] tensor(float)
outputs: ['raw_predictions']
frames_measured: 100
crop_top: None
output_shapes: [(1, 192, 78)]
preprocess_ms_mean: 0.00
preprocess_ms_p95: 0.00
inference_ms_mean: 699.92
inference_ms_p95: 700.59
total_ms_mean: 700.28
total_ms_p95: 700.95
fps_from_mean_total: 1.43

[stderr]
2026-04-17 02:52:05.890130072 [W:onnxruntime:Default, device_discovery.cc:283 GetGpuDevices] Failed to detect devices under "/sys/class/drm/card1": device_discovery.cc:93 ReadFileContents Failed to open file: "/sys/class/drm/card1/device/vendor"
2026-04-17 02:52:05.890175757 [W:onnxruntime:Default, device_discove

## 2차 실행 — Camera 포함 속도

dummy가 모델 계산만 본다면, 이 셀은 실제 camera capture 이후 preprocess+inference까지 포함한다.

이 값이 실제 lane loop의 체감 속도에 더 가깝다.

In [6]:
BEST_THREADS = 4
cmd = [
    sys.executable, str(SCRIPT),
    "--model", str(MODEL),
    "--camera",
    "--frames", "100",
    "--warmup", "10",
    "--threads", str(BEST_THREADS),
]
camera_result = run_cmd(cmd)

$ /home/pi/AI_CAR/env/bin/python3 /home/pi/AI_CAR/architecture_validation/lane_model/scripts/03_lane_onnx_bench.py --model /home/pi/AI_CAR/architecture_validation/lane_model/models/clrkdnet_resnet18_culane.onnx --camera --frames 100 --warmup 10 --threads 4
model: /home/pi/AI_CAR/architecture_validation/lane_model/models/clrkdnet_resnet18_culane.onnx
input: image ['batch', 3, 320, 800] tensor(float)
outputs: ['raw_predictions']
frames_measured: 100
crop_top: 445
output_shapes: [(1, 192, 78)]
preprocess_ms_mean: 4.29
preprocess_ms_p95: 6.77
inference_ms_mean: 388.74
inference_ms_p95: 397.05
total_ms_mean: 399.14
total_ms_p95: 409.47
fps_from_mean_total: 2.51

[stderr]
2026-04-17 02:56:59.362120231 [W:onnxruntime:Default, device_discovery.cc:283 GetGpuDevices] Failed to detect devices under "/sys/class/drm/card1": device_discovery.cc:93 ReadFileContents Failed to open file: "/sys/class/drm/card1/device/vendor"
2026-04-17 02:56:59.362168009 [W:onnxruntime:Default, device_discovery.cc:283 G

## 판단 기준

| FPS | 판단 |
|---:|---|
| 15 이상 | lane model 단독 후보로 충분 |
| 10~15 | 저속 주행 후보 |
| 5~10 | 위험. fallback 필요 |
| 5 미만 | CLRKDNet 단독 주행은 매우 위험 |

## Log 03 초안 메모

```markdown
ONNX export가 성공했기 때문에 다음 질문은 속도였다. Pi에서 모델이 돌아가는 것과 주행에 충분히 빠른 것은 다르다.

`03_lane_onnx_bench.py`로 dummy input과 camera input을 나누어 측정했다. dummy는 모델 자체의 연산 속도를 보기 위한 것이고, camera는 실제 입력 파이프라인까지 포함한 속도를 보기 위한 것이다.

결과:
- 가장 빠른 thread 수 = 
- dummy fps = 
- camera fps = 

이 결과는 ... 를 의미한다. 따라서 CLRKDNet 단독 구조는 ... 로 판단한다.
```